In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as f

In [0]:
dbutils.widgets.text("catalogo","catalog_au")
dbutils.widgets.text("esquema_source","silver")
dbutils.widgets.text("esquema_sink","golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_sms_silver = spark.table(f"{catalogo}.{esquema_source}.sms_transformed")
df_email_silver = spark.table(f"{catalogo}.{esquema_source}.emails_transformed")

In [0]:
df_sms_golden = df_sms_silver.groupBy("anio", "mes", "periodo", "cliente", "producto", "usuario_creacion").agg(count("usuario_creacion").cast("int").alias("total_sms"))


In [0]:
df_sms_golden = df_sms_golden.select("anio", "mes","cliente", "producto", "usuario_creacion", "total_sms", "periodo")

In [0]:
df_email_golden = df_email_silver.groupBy("anio", "mes", "cliente", "producto","senderaddress").agg(
 count(when(col("Status") == "Delivered", 1)).cast("int").alias("entregado"),\
 count(when(col("Status") == "Failed", 1)).cast("int").alias("fallido"),\
 count("*").cast("int").alias("total"))                                                                                               

In [0]:
df_sms_golden.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.golden_sms")
df_email_golden.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.golden_email")

In [0]:
%sql
SELECT * FROM catalog_au.golden.golden_sms

In [0]:
%sql
SELECT * FROM catalog_au.golden.golden_email